# Dask on amlhpc

Run a Dask scheduler on this Compute Instance, attach AmlCompute worker VMs with `sbatch`, and run a real distributed calculation.

Before submitting workers from a terminal, connect that terminal to this workspace by sourcing the cluster profile written by `deploy init`:

```bash
source ~/.amlhpc/<clustername>.sh   # restores SUBSCRIPTION, CI_RESOURCE_GROUP, CI_WORKSPACE
sinfo                               # confirm the connection and see partitions
```

Use the `Python 3.8 - AzureML` kernel for this notebook (Dask is preinstalled there).

In [ ]:
from dask.distributed import Client, LocalCluster, progress
client = Client(LocalCluster(ip='0.0.0.0', scheduler_port=12345, n_workers=0))
client

Read the scheduler address off the `Scheduler` box above (the `comm` field, e.g. `tcp://10.0.1.5:12345`) and put it on the last line of `dask.job`. Then, from a terminal where you have sourced the cluster profile, add 4 worker VMs:

```bash
sbatch -p f4s --array 1-4 ./dask.job
```

Watch them register in the client widget above (or the Dask dashboard) before running the calculation below.

## A real calculation: estimate π with distributed Monte Carlo

Throw random darts at the unit square; the fraction landing inside the quarter circle approximates π/4. This is embarrassingly parallel, so every worker VM does an independent chunk and we reduce the counts. The more samples (and workers) you add, the closer the result gets to π.

In [ ]:
import random

def count_inside(n_samples, seed):
    rng = random.Random(seed)
    inside = 0
    for _ in range(n_samples):
        x, y = rng.random(), rng.random()
        if x * x + y * y <= 1.0:
            inside += 1
    return inside

In [ ]:
%%time

n_tasks = 512
samples_per_task = 2_000_000
total_samples = n_tasks * samples_per_task

futures = [client.submit(count_inside, samples_per_task, seed) for seed in range(n_tasks)]
progress(futures)

inside_total = sum(client.gather(futures))
pi_estimate = 4.0 * inside_total / total_samples

print(f"samples : {total_samples:,}")
print(f"pi ~    : {pi_estimate}")
print(f"error   : {abs(pi_estimate - 3.141592653589793):.2e}")

Shut the scheduler and all worker VMs down. This finalizes the jobs and deallocates the AmlCompute workers so you stop paying for them.

In [ ]:
client.shutdown()